# 02 — Reading and writing GRAND data

Everything GRAND records or simulates lives in ROOT `TTree`s, and
`grand.dataio` is the layer that reads and writes them. This notebook builds a
file from nothing, reads it back, and works through the conventions that govern
how files are grouped — which are not written down anywhere else and which I
got wrong three times while testing them.

The long form of the [data model page](https://grand-mother.github.io/grand-docs/datamodel.html).

In [1]:
import tempfile, os
import numpy as np

from grand.dataio.run_trees import TRun
from grand.dataio.event_trees import TEfield, TVoltage
from grand.dataio.data_handling import DataDirectory, DataFile

workdir = tempfile.mkdtemp()
print("working in", workdir)

working in /tmp/tmpnoau51a7


## 1. The naming convention

The class names encode what a tree holds, and the convention is worth learning
before anything else:

| Pattern | Meaning |
|---|---|
| `TRun*` | constant for the duration of a run — one entry per run |
| `T*` | one entry per event |
| `T*Sim` | produced **only** by simulators |
| `T*` without `Sim` | hardware-origin; simulation may fill it, leaving unavailable fields empty |

So `TShower` is what a detector would have recorded, and `TShowerSim` is what
only a simulator knows.

In [2]:
path = os.path.join(workdir, 'efield_20260101_000000_RUN0_L0_0000.root')

run = TRun(path)
run.run_number = 0
run.du_id = [0, 1, 2]
run.du_xyz = [[0., 0., 0.], [1000., 0., 0.], [0., 1000., 0.]]
run.t_bin_size = [0.5] * 3          # nanoseconds
run.origin_geoid = [40.98, 93.95, 1200.0]
run.analysis_level = 0
run.fill()
run.write()
print("wrote a run tree with", len(run.du_id), "detection units")

No valid trun TTree in the file /tmp/tmpnoau51a7/efield_20260101_000000_RUN0_L0_0000.root. Creating a new one.


wrote a run tree with 3 detection units


## 2. Fields are descriptors, not plain attributes

A field on a tree class is declared like this:

```python
nutrig_rhox: StdVectorListDesc = field(default=StdVectorListDesc("unsigned short"))
```

`StdVectorListDesc` is a descriptor that translates between Python values and
the C++ type the branch holds. **Adding a field adds a branch to the on-disk
format**, which is why a field addition is a change to a data contract rather
than an implementation detail — and why `tests/dataio/test_schema_snapshot.py`
exists to make such changes visible in a diff.

In [3]:
t = np.arange(256) * 0.5                    # ns
efield = TEfield(path)
for event in range(3):
    pulse = np.exp(-((t - (40.0 + 20.0 * event)) ** 2) / (2 * 4.0 ** 2))
    efield.run_number, efield.event_number = 0, event
    efield.du_id = [0, 1, 2]
    efield.du_nanoseconds = [0, 0, 0]
    efield.du_seconds = [0, 0, 0]
    efield.trace = np.stack([np.stack([pulse, .5 * pulse, .2 * pulse])] * 3).astype(np.float32)
    efield.analysis_level = 0
    efield.fill()
efield.write()

print("events written:", len(efield.get_list_of_events()))

No valid tefield TTree in the file /tmp/tmpnoau51a7/efield_20260101_000000_RUN0_L0_0000.root. Creating a new one.


events written: 3


## 3. Reading it back

In [4]:
handle = DataFile(path)
print("trees exposed as attributes:",
      [a for a in ('trun', 'tefield', 'tvoltage', 'tadc') if hasattr(handle, a)])

handle.tefield.get_entry(1)
trace = np.asarray(handle.tefield.trace)
print("event 1 trace shape (units, arms, samples):", trace.shape)
print("peak sample per unit:", [int(np.argmax(trace[i, 0])) for i in range(trace.shape[0])])

trees exposed as attributes: ['trun', 'tefield']
event 1 trace shape (units, arms, samples): (3, 3, 256)
peak sample per unit: [120, 120, 120]


## 4. How files are grouped — three traps

`DataDirectory` scans a directory and decides which files belong together. Its
rules are not documented anywhere, and each of them can bite.

**Grouping keys on tree type and analysis level, not on run number.** That is
correct for the layout the converters produce — `sim2root` writes one run per
directory — but it means pointing `DataDirectory` at a directory holding
several runs *silently merges them*.

In [5]:
directory = DataDirectory(workdir)
print("files found  :", [os.path.basename(f) for f in directory.get_list_of_files()])
print("handles      :", len(directory.get_list_of_files_handles()))

files found  : ['efield_20260101_000000_RUN0_L0_0000.root']
handles      : 1


**The level in the filename must match the level inside the trees.** The
scanner takes the analysis level from the `_L0_`/`_L1_` marker in the name, then
looks for a tree attribute named for the level recorded *in the tree*. When
they disagree you get `AttributeError: 'DataFile' object has no attribute
'tefield_l1'` — naming something you never wrote, and saying nothing about the
real cause.

In [6]:
bad = os.path.join(workdir, 'mismatch_20260101_000000_RUN0_L1_0000.root')
r = TRun(bad); r.run_number = 0; r.du_id = [0]; r.du_xyz = [[0., 0., 0.]]
r.t_bin_size = [0.5]; r.analysis_level = 0          # name says L1, tree says 0
r.fill(); r.write()

try:
    DataDirectory(workdir).get_list_of_files_handles()
except AttributeError as exc:
    print("AttributeError:", exc)

No valid trun TTree in the file /tmp/tmpnoau51a7/mismatch_20260101_000000_RUN0_L1_0000.root. Creating a new one.


**Both levels are returned, but the bare attribute follows the highest.** With
an L0 and an L1 file present, `directory.tefield` refers to L1 while
`tefield_l0` and `tefield_l1` name them individually. A script reading
`directory.tefield` therefore changes behaviour the moment someone drops an L1
file beside the L0 one.

## 5. Provenance

`TRun` carries `software_version`, `analysis_level`, `site` and `site_layout`.
This matters more than it sounds: a change to the Galactic-noise normalisation
alters every voltage in a file without changing its shape, and the version
stamp is the only way to tell two such files apart.

In [7]:
handle.trun.get_entry(0)
for field_name in ('run_number', 'site', 'site_layout', 'analysis_level'):
    print("%-16s %r" % (field_name, getattr(handle.trun, field_name, None)))

run_number       np.uint32(0)
site             ''
site_layout      ''
analysis_level   0


## Where next

- [01 — Coordinate systems](01_coordinates.ipynb)
- [03 — The antenna response](03_antenna_response.ipynb)
- The [API reference](https://grand-mother.github.io/grand-docs/api.html)